# Chapter 13 — Token Counts Don't Add Up

**Companion to *Applied AI*.**

This notebook accompanies Chapter 13. Before you add, compare, price or route
on the usage numbers a provider reports, you have to decide what each one
means.

The chapter's interpreter was run as a pure projection over four **real**
preserved responses and twenty synthetic edge cases, with expectations written
by hand from the dialect documentation *before* any code ran. That run is
preserved here.

## Question

**Do these usage fields share a unit?**

## What this notebook establishes

- The four real responses re-read from the preserved report: the same request
  counted as 38, 279 and 73 input tokens depending on the route.
- `usage-semantics-v1` (two numbers) beside `v2` (components, relations,
  derived quantities, UNKNOWNs) on identical bytes.
- The Messages route where the total input is **UNKNOWN with a lower bound**,
  because the dialect adds cache to input and the payload omitted the cache
  fields.
- Two real diagnostics: a reported reasoning count of **zero beside visible
  reasoning content**, and reasoning content with no count at all.
- What zero-filling does to a cost estimate, and why it biases downward exactly
  where accuracy matters most.

## What this notebook does **not** establish

- Normalisation makes **one route's** usage internally coherent. It does not
  make two routes' token counts one unit, and this notebook does not claim it
  does.
- No bill is computed. The chapter stops deliberately at interpretation:
  accounting needs a pricing *shape* (free-with-conditions, subscription with
  caps, or per-token) that a rate table cannot represent.
- The Chat rule is applied **by analogy** to documented Responses semantics, and
  route conformance is unverified. The evidence says so in its own rule source.

## Setup

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="usage-semantics"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError("Set APPLIED_AI_EVIDENCE to the evidence directory.")

EVIDENCE_DIR = find_evidence_dir()
report = json.loads(
    (EVIDENCE_DIR / "usage-semantics" / "offline" / "report.json").read_text(encoding="utf-8"))

print("versions      :", report["versions"])
print("network calls :", report["network_calls"])
print("real cases    :", report["real_cases"])
print("synthetic     :", report["synthetic_cases"])
print("mismatches vs hand-written expectations:", report["mismatch_count"])
print()
print("Expectations were written by hand from the dialect documentation BEFORE")
print("the run; that file's hash is recorded in the report.")

versions      : ['usage-semantics-v1', 'usage-semantics-v2']
network calls : 0
real cases    : 4
synthetic     : 20
mismatches vs hand-written expectations: 0

Expectations were written by hand from the dialect documentation BEFORE
the run; that file's hash is recorded in the report.


## 1. Three numbers that will not add

The same small task, through the same gateway, all three succeeded.

In [2]:
real = [r for r in report["rows"] if r["evidence_class"] == "transport_body"]

print(f"{'case':<28}{'protocol':<20}{'v1 input':>10}{'v1 output':>11}")
print("-" * 70)
for r in real:
    vi, vo = r["summary"]["v1"]
    print(f"{r['case']:<28}{r['protocol']:<20}{vi:>10}{vo:>11}")

inputs = [r["summary"]["v1"][0] for r in real]
print()
print(f"input counts reported: {inputs}")
print()
print("Put those in a spreadsheet and the next move writes itself: sum them,")
print("average them, divide the invoice by them, route to whichever chamber is")
print("'cheapest per token'. Every one of those moves assumes they measure the")
print("same thing.")

case                        protocol              v1 input  v1 output
----------------------------------------------------------------------
stage12-responses-luna      responses                   38         26
stage12-chat-mimo           chat_completions           279        166
stage12-messages-minimax    messages                    73        220

input counts reported: [38, 279, 73]

Put those in a spreadsheet and the next move writes itself: sum them,
average them, divide the invoice by them, route to whichever chamber is
'cheapest per token'. Every one of those moves assumes they measure the
same thing.


## 2. A name is not a unit

The two dialect families, from their own documentation.

In [3]:
RULES = {
    "OpenAI Responses":  {"cached inside input_tokens?": "YES",
                          "cache reported at": "input_tokens_details.cached_tokens",
                          "total reported?": "total_tokens"},
    "Anthropic Messages": {"cached inside input_tokens?": "NO (added beside)",
                           "cache reported at": "cache_read_input_tokens",
                           "total reported?": "none"},
}
keys = list(next(iter(RULES.values())))
print(f"{'question':<34}" + "".join(f"{k:<38}" for k in RULES))
print("-" * 112)
for k in keys:
    print(f"{k:<34}" + "".join(f"{RULES[d][k]:<38}" for d in RULES))

print()
print("The SAME field name, input_tokens, has OPPOSITE relationships to cached")
print("input in the two dialects. Code that reads it from both and adds them is")
print("not normalising - it is the Mars Climate Orbiter's ground software.")

question                          OpenAI Responses                      Anthropic Messages                    
----------------------------------------------------------------------------------------------------------------
cached inside input_tokens?       YES                                   NO (added beside)                     
cache reported at                 input_tokens_details.cached_tokens    cache_read_input_tokens               
total reported?                   total_tokens                          none                                  

The SAME field name, input_tokens, has OPPOSITE relationships to cached
input in the two dialects. Code that reads it from both and adds them is
not normalising - it is the Mars Climate Orbiter's ground software.


## 3. v1 against v2, on identical bytes

`v1` reproduces the historical adapter view: two numbers. `v2` applies the
four rules. Both are pure functions of the same preserved observation.

The pairs below are `[value, lower_bound]`. A `None` value means **UNKNOWN**.

In [4]:
def fmt(pair):
    v, lb = pair
    if v is None:
        return f"UNKNOWN (>= {lb})"
    return str(v)

print(f"{'case':<28}{'v1 in/out':<14}{'total_input':<20}"
      f"{'fresh_input':<14}{'processed_total'}")
print("-" * 96)
for r in real:
    s = r["summary"]
    vi, vo = s["v1"]
    print(f"{r['case']:<28}{f'{vi}/{vo}':<14}{fmt(s['total_input']):<20}"
          f"{fmt(s['fresh_input']):<14}{fmt(s['processed_total'])}")

case                        v1 in/out     total_input         fresh_input   processed_total
------------------------------------------------------------------------------------------------
stage12-responses-luna      38/26         38                  38            64
stage12-chat-mimo           279/166       279                 87            445
stage12-messages-minimax    73/220        UNKNOWN (>= 73)     73            UNKNOWN (>= 293)


## Observation

Read the `fresh_input` column, route by route.

In [5]:
for r in real:
    s = r["summary"]
    vi, _ = s["v1"]
    fresh = s["fresh_input"][0]
    if r["protocol"] == "responses":
        note = "no cache component reported, so fresh == input"
        cached = 0
    elif r["protocol"] == "chat_completions":
        cached = vi - fresh
        note = f"{cached} cached tokens were INSIDE the {vi}-token input count"
    else:
        cached = None
        note = "input_tokens EXCLUDES cache by rule; cache fields were absent"
    print(f"{r['case']:<28} v1 input {vi:>4}  ->  fresh {fresh:>4}   {note}")

fresh_values = [r["summary"]["fresh_input"][0] for r in real]
print()
print("fresh input across the four cases:", fresh_values)
assert fresh_values[:3] == [38, 87, 73]
print()
print("assertion held: reproduces the chapter's 38 / 87 / 73")
print("(the fourth case is Chapter 11's live call: 264 input, 192 cached -> 72)")

stage12-responses-luna       v1 input   38  ->  fresh   38   no cache component reported, so fresh == input
stage12-chat-mimo            v1 input  279  ->  fresh   87   192 cached tokens were INSIDE the 279-token input count
stage12-messages-minimax     v1 input   73  ->  fresh   73   input_tokens EXCLUDES cache by rule; cache fields were absent

fresh input across the four cases: [38, 87, 73]

assertion held: reproduces the chapter's 38 / 87 / 73
(the fourth case is Chapter 11's live call: 264 input, 192 cached -> 72)


## 4. What did not become a number

Under the Messages rule, cache reads and writes are **added** to input. The
MiniMax route left both fields out of the payload.

In [6]:
msg = next(r for r in real if r["protocol"] == "messages")
s = msg["summary"]

print("Messages route:")
print(f"  v1 said                : input = {s['v1'][0]}")
print(f"  v2 total_input         : {fmt(s['total_input'])}")
print(f"  v2 processed_total     : {fmt(s['processed_total'])}")
print()
print("v1 would have given you 73 and let you believe it was the whole input.")
print("v2 gives you 73 as a FLOOR and says so.")
print()
print("That is the difference between a normaliser and a renamer. A renamer")
print("maps prompt_tokens to input_tokens and moves on. An interpreter asks")
print("what is inside the number first.")

Messages route:
  v1 said                : input = 73
  v2 total_input         : UNKNOWN (>= 73)
  v2 processed_total     : UNKNOWN (>= 293)

v1 would have given you 73 and let you believe it was the whole input.
v2 gives you 73 as a FLOOR and says so.

That is the difference between a normaliser and a renamer. A renamer
maps prompt_tokens to input_tokens and moves on. An interpreter asks
what is inside the number first.


## 5. What v2 caught that v1 could not express

In [7]:
print(f"{'case':<28}{'diagnostics / unrecognised paths'}")
print("-" * 96)
for r in real:
    s = r["summary"]
    notes = list(s["diagnostics"])
    for p in s["unrecognized_paths"]:
        notes.append(f"unrecognised: {p.split('.')[-1]} (kept, not interpreted)")
    print(f"{r['case']:<28}{notes[0] if notes else '-'}")
    for extra in notes[1:]:
        print(f"{'':<28}{extra}")

print()
print("provider 'cost' field on every one of the four responses:",
      {r["summary"]["provider_cost_raw"] for r in real})
print("The same string appears on free models and subscription models alike,")
print("so on its own it is not a price. v2 preserves it and marks accounting")
print("unknown.")

case                        diagnostics / unrecognised paths
------------------------------------------------------------------------------------------------
stage12-responses-luna      -
stage12-chat-mimo           reasoning_tokens_zero_with_reasoning_content
                            unrecognised: audio_tokens (kept, not interpreted)
                            unrecognised: audio_tokens (kept, not interpreted)
stage12-messages-minimax    reasoning_content_present_reasoning_tokens_not_reported

provider 'cost' field on every one of the four responses: {'0'}
The same string appears on free models and subscription models alike,
so on its own it is not a price. v2 preserves it and marks accounting
unknown.


The first diagnostic is the load-bearing one.

> `reasoning_tokens_zero_with_reasoning_content`

Two MiMo calls reported `reasoning_tokens: 0` while returning reasoning
content. **v2 does not "correct" the zero**, because it has no evidence for the
right count. It records the discrepancy, preserves the reported zero, and marks
route conformance unverified.

A payload that does not fit the assumed mapping is **evidence to retain, not a
number to repair**.

## 6. Absence is not zero

The chapter's rule that runs through all four. Price the Messages route two
ways.

In [8]:
PRICE_FRESH_PER_1K   = 1.00      # illustrative
PRICE_CACHED_PER_1K  = 0.10      # cached reads are typically an order cheaper
PRICE_OUTPUT_PER_1K  = 4.00

v1_in, v1_out = msg["summary"]["v1"]

# The naive path: treat the reported input as the whole input, cache as zero.
naive_cost = (v1_in * PRICE_FRESH_PER_1K + v1_out * PRICE_OUTPUT_PER_1K) / 1000

# The honest path: the total input is unknown, so the bill is a lower bound.
floor_cost = naive_cost
print(f"zero-filled 'cost' : ${naive_cost:.6f}   <- looks precise")
print(f"honest statement   : at least ${floor_cost:.6f}, total input UNKNOWN")
print()
print("Both use the same arithmetic. Only one of them is a claim you can")
print("defend when the route later reports a cache read.")

zero-filled 'cost' : $0.953000   <- looks precise
honest statement   : at least $0.953000, total input UNKNOWN

Both use the same arithmetic. Only one of them is a claim you can
defend when the route later reports a cache read.


The chapter's reason this matters is statistical, not aesthetic. Usage goes
missing most often on **failed** attempts — timeouts, rejected requests,
dropped connections. Filling those with zero biases every cost figure downward,
worst exactly where failures are most frequent.

In [9]:
# A small illustration of that bias. Invented numbers.
attempts = [
    ("succeeded", 264), ("succeeded", 251), ("timeout", None),
    ("succeeded", 240), ("rejected", None), ("succeeded", 259),
    ("timeout", None),
]
observed = [t for _, t in attempts if t is not None]
zero_filled = [t or 0 for _, t in attempts]

print(f"attempts            : {len(attempts)}")
print(f"usage reported for  : {len(observed)}")
print(f"mean over reported  : {sum(observed)/len(observed):.1f} tokens")
print(f"mean if zero-filled : {sum(zero_filled)/len(zero_filled):.1f} tokens")
print(f"understatement      : {1 - (sum(zero_filled)/len(zero_filled))/(sum(observed)/len(observed)):.0%}")
print()
print("Zero is a measurement. 'Nothing was reported' is not.")

attempts            : 7
usage reported for  : 4
mean over reported  : 253.5 tokens
mean if zero-filled : 144.9 tokens
understatement      : 43%

Zero is a measurement. 'Nothing was reported' is not.


## 7. The limit of normalisation

In [10]:
print("fresh input, route by route:", fresh_values[:3])
print()
print("It is tempting to read those as three comparable counts. They are not.")
print()
print("Each came from a different route serving a different MODEL. The")
print("preserved responses establish what each route reported. They do not")
print("establish that all three used the same tokenizer, the same")
print("preprocessing, or an identical token unit.")
print()
print("So normalisation makes a route's usage INTERNALLY COHERENT.")
print("It cannot make two routes' token counts share one unit.")
print()
print("That is why Chapter 10 compares chambers on cost per PASSING ITEM")
print("rather than cost per token.")

fresh input, route by route: [38, 87, 73]

It is tempting to read those as three comparable counts. They are not.

Each came from a different route serving a different MODEL. The
preserved responses establish what each route reported. They do not
establish that all three used the same tokenizer, the same
preprocessing, or an identical token unit.

So normalisation makes a route's usage INTERNALLY COHERENT.
It cannot make two routes' token counts share one unit.

That is why Chapter 10 compares chambers on cost per PASSING ITEM
rather than cost per token.


## Interpretation

The chapter's three layers, and where this notebook stops:

```text
observation     the preserved response bytes, content-addressed   never rewritten
     v
interpretation  usage-semantics-v2: components, relations,        versioned, replaceable
                derived values, conflicts, diagnostics, UNKNOWNs
     v
accounting      a bill, a quota draw-down, a budget charge        NOT BUILT
```

Accounting needs things the interpreter does not have and should not guess: the
pricing shape of the plan, the multiplier for each component, and an effective
date. `cost` stays UNKNOWN until a separately justified pricing interpretation
exists.

The versioning is what makes this safe. `v1` is not deleted, because decisions
already made were made under it. When a dialect's documentation changes, the
fix is `v3` over the same bytes, and history does not move.

## Try it yourself

1. **Write v3.** Decide that a Chat route's `reasoning_tokens: 0` beside
   visible reasoning content should be recorded as UNKNOWN rather than zero.
   Apply it to the same rows. Which derived quantities change, and which
   historical decisions would you have to leave alone?
2. **Find the Messages cache trap.** The chapter notes that CodeAI's Messages
   codec reads only `input_tokens`. Construct a synthetic Messages payload
   *with* `cache_read_input_tokens` and work out what the canonical field would
   then mean.
3. **Load the synthetic corpus.** The twenty other rows in this report cover
   absent usage, measured zeros, conflicting totals, negative and string counts,
   and a protocol with no rule at all. Print the rows where `conflicts` is
   non-empty.
4. **Search your own code** for the point where "not reported" becomes `0`.
   The chapter says there is almost always one.